# sPHENIX 3D IBF field QA

This notebook reads the merged 3D ROOT field map and produces QA plots for the electric-field components and, when available, the 3D charge-density map.

Expected objects:

- `Field3D/hEx`
- `Field3D/hEy`
- `Field3D/hEz`
- optional `Field3D/hPhi`
- optional `Charge3D/hRho`


In [ ]:

import ROOT
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

ROOT.gROOT.SetBatch(True)

INPUT_ROOT = "sphenix_3d_ibf_field.root"
OUTPUT_DIR = Path("ibf_qa_plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_PLOTS = True
SHOW_PLOTS = True

# Representative slice positions.
R_SLICE_CM = 50.0
PHI_SLICE_RAD = 0.0
Z_SLICE_CM = 0.0

# Longitudinal profile radius and phi.
PROFILE_R_CM = 50.0
PROFILE_PHI_RAD = 0.0

# Radial profile phi and z.
RADIAL_PROFILE_PHI_RAD = 0.0
RADIAL_PROFILE_Z_CM = 0.0

# Phi profile radius and z.
PHI_PROFILE_R_CM = 50.0
PHI_PROFILE_Z_CM = 0.0


## ROOT helpers


In [ ]:

def require_hist3(root_file, path):
    obj = root_file.Get(path)
    if not obj:
        raise RuntimeError(f"Missing object: {path}")
    if not obj.InheritsFrom("TH3"):
        raise TypeError(f"{path} is not a TH3")
    obj.SetDirectory(0)
    return obj


def optional_hist3(root_file, path):
    obj = root_file.Get(path)
    if not obj:
        return None
    if not obj.InheritsFrom("TH3"):
        raise TypeError(f"{path} is not a TH3")
    obj.SetDirectory(0)
    return obj


def axis_edges(axis):
    nbins = axis.GetNbins()
    edges = np.empty(nbins + 1, dtype=float)

    for i in range(1, nbins + 1):
        edges[i - 1] = axis.GetBinLowEdge(i)

    edges[-1] = axis.GetBinUpEdge(nbins)
    return edges


def axis_centers(axis):
    return np.array(
        [axis.GetBinCenter(i) for i in range(1, axis.GetNbins() + 1)],
        dtype=float,
    )


def th3_to_numpy(hist):
    nx = hist.GetNbinsX()
    ny = hist.GetNbinsY()
    nz = hist.GetNbinsZ()

    arr = np.empty((nx, ny, nz), dtype=float)

    for ix in range(nx):
        for iy in range(ny):
            for iz in range(nz):
                arr[ix, iy, iz] = hist.GetBinContent(
                    ix + 1,
                    iy + 1,
                    iz + 1,
                )

    x_edges = axis_edges(hist.GetXaxis())
    y_edges = axis_edges(hist.GetYaxis())
    z_edges = axis_edges(hist.GetZaxis())

    return arr, x_edges, y_edges, z_edges


def nearest_bin(centers, value):
    return int(np.argmin(np.abs(centers - value)))


def save_or_show(fig, name):
    if SAVE_PLOTS:
        fig.savefig(
            OUTPUT_DIR / f"{name}.png",
            dpi=180,
            bbox_inches="tight",
        )

    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


def symmetric_limit(data, percentile=99.5):
    finite = np.asarray(data)[np.isfinite(data)]

    if finite.size == 0:
        return 1.0

    limit = np.percentile(np.abs(finite), percentile)

    if limit <= 0.0:
        limit = np.max(np.abs(finite))

    return limit if limit > 0.0 else 1.0


## Load field and charge maps


In [ ]:

input_path = Path(INPUT_ROOT)

if not input_path.exists():
    raise FileNotFoundError(
        f"Could not find {input_path.resolve()}"
    )

root_file = ROOT.TFile.Open(str(input_path), "READ")

if not root_file or root_file.IsZombie():
    raise RuntimeError(f"Could not open {input_path}")

h_ex = require_hist3(root_file, "Field3D/hEx")
h_ey = require_hist3(root_file, "Field3D/hEy")
h_ez = require_hist3(root_file, "Field3D/hEz")

h_phi = optional_hist3(root_file, "Field3D/hPhi")
h_rho = optional_hist3(root_file, "Charge3D/hRho")

Ex, r_edges, phi_edges, z_edges = th3_to_numpy(h_ex)
Ey, _, _, _ = th3_to_numpy(h_ey)
Ez, _, _, _ = th3_to_numpy(h_ez)

Phi = None
if h_phi is not None:
    Phi, _, _, _ = th3_to_numpy(h_phi)

rho = None
rho_edges = None
if h_rho is not None:
    rho, rho_r_edges, rho_phi_edges, rho_z_edges = th3_to_numpy(h_rho)
    rho_edges = (
        rho_r_edges,
        rho_phi_edges,
        rho_z_edges,
    )

root_file.Close()

r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
phi_centers = 0.5 * (phi_edges[:-1] + phi_edges[1:])
z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])

Eabs = np.sqrt(Ex**2 + Ey**2 + Ez**2)

R_grid, Phi_grid, Z_grid = np.meshgrid(
    r_centers,
    phi_centers,
    z_centers,
    indexing="ij",
)

Er = Ex * np.cos(Phi_grid) + Ey * np.sin(Phi_grid)
Ephi = -Ex * np.sin(Phi_grid) + Ey * np.cos(Phi_grid)

print("Input:", input_path.resolve())
print("Field shape:", Ex.shape)
print("r range [cm]:", r_edges[0], r_edges[-1])
print("phi range [rad]:", phi_edges[0], phi_edges[-1])
print("z range [cm]:", z_edges[0], z_edges[-1])

print("Maximum |Ex| [V/m]:", np.max(np.abs(Ex)))
print("Maximum |Ey| [V/m]:", np.max(np.abs(Ey)))
print("Maximum |Ez| [V/m]:", np.max(np.abs(Ez)))
print("Maximum |E|  [V/m]:", np.max(Eabs))

if rho is not None:
    print("Charge map shape:", rho.shape)
    print("Total stored charge density sum:", np.sum(rho))
else:
    print("No Charge3D/hRho found; charge QA will be skipped.")


## Global distributions


In [ ]:

components = {
    "Ex": Ex,
    "Ey": Ey,
    "Ez": Ez,
    "Er": Er,
    "Ephi": Ephi,
    "Eabs": Eabs,
}

for name, array in components.items():
    fig, ax = plt.subplots(figsize=(8, 5))

    values = array[np.isfinite(array)].ravel()

    ax.hist(
        values,
        bins=150,
        histtype="step",
    )

    ax.set_xlabel(f"{name} [V/m]")
    ax.set_ylabel("Bins")
    ax.set_title(f"Global distribution of {name}")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)

    save_or_show(fig, f"global_distribution_{name}")


## \(r\)-\(\phi\) slices at fixed \(z\)


In [ ]:

iz = nearest_bin(z_centers, Z_SLICE_CM)
actual_z = z_centers[iz]

for name, array in components.items():
    data = array[:, :, iz]

    fig, ax = plt.subplots(figsize=(9, 6))

    if name == "Eabs":
        mesh = ax.pcolormesh(
            r_edges,
            phi_edges,
            data.T,
            shading="auto",
        )
    else:
        limit = symmetric_limit(data)
        mesh = ax.pcolormesh(
            r_edges,
            phi_edges,
            data.T,
            shading="auto",
            vmin=-limit,
            vmax=limit,
            cmap="coolwarm",
        )

    fig.colorbar(mesh, ax=ax, label=f"{name} [V/m]")
    ax.set_xlabel("r [cm]")
    ax.set_ylabel(r"$\phi$ [rad]")
    ax.set_title(
        rf"{name} in $r$-$\phi$ at z={actual_z:.2f} cm"
    )

    save_or_show(
        fig,
        f"rphi_{name}_z_{actual_z:+.2f}",
    )


## \(r\)-\(z\) slices at fixed \(\phi\)


In [ ]:

iphi = nearest_bin(phi_centers, PHI_SLICE_RAD)
actual_phi = phi_centers[iphi]

for name, array in components.items():
    data = array[:, iphi, :]

    fig, ax = plt.subplots(figsize=(9, 6))

    if name == "Eabs":
        mesh = ax.pcolormesh(
            r_edges,
            z_edges,
            data.T,
            shading="auto",
        )
    else:
        limit = symmetric_limit(data)
        mesh = ax.pcolormesh(
            r_edges,
            z_edges,
            data.T,
            shading="auto",
            vmin=-limit,
            vmax=limit,
            cmap="coolwarm",
        )

    fig.colorbar(mesh, ax=ax, label=f"{name} [V/m]")
    ax.set_xlabel("r [cm]")
    ax.set_ylabel("z [cm]")
    ax.set_title(
        rf"{name} in $r$-$z$ at $\phi$={actual_phi:.3f}"
    )

    save_or_show(
        fig,
        f"rz_{name}_phi_{actual_phi:+.3f}",
    )


## \(\phi\)-\(z\) slices at fixed \(r\)


In [ ]:

ir = nearest_bin(r_centers, R_SLICE_CM)
actual_r = r_centers[ir]

for name, array in components.items():
    data = array[ir, :, :]

    fig, ax = plt.subplots(figsize=(9, 6))

    if name == "Eabs":
        mesh = ax.pcolormesh(
            phi_edges,
            z_edges,
            data.T,
            shading="auto",
        )
    else:
        limit = symmetric_limit(data)
        mesh = ax.pcolormesh(
            phi_edges,
            z_edges,
            data.T,
            shading="auto",
            vmin=-limit,
            vmax=limit,
            cmap="coolwarm",
        )

    fig.colorbar(mesh, ax=ax, label=f"{name} [V/m]")
    ax.set_xlabel(r"$\phi$ [rad]")
    ax.set_ylabel("z [cm]")
    ax.set_title(
        rf"{name} in $\phi$-$z$ at r={actual_r:.2f} cm"
    )

    save_or_show(
        fig,
        f"phiz_{name}_r_{actual_r:.2f}",
    )


## One-dimensional field profiles


In [ ]:

# z profiles
ir = nearest_bin(r_centers, PROFILE_R_CM)
iphi = nearest_bin(phi_centers, PROFILE_PHI_RAD)

fig, ax = plt.subplots(figsize=(9, 6))
for name, array in components.items():
    if name == "Eabs":
        continue
    ax.plot(
        z_centers,
        array[ir, iphi, :],
        label=name,
    )

ax.set_xlabel("z [cm]")
ax.set_ylabel("Field [V/m]")
ax.set_title(
    rf"Field vs z at r={r_centers[ir]:.2f} cm, "
    rf"$\phi$={phi_centers[iphi]:.3f}"
)
ax.axhline(0.0, linewidth=1.0)
ax.grid(alpha=0.25)
ax.legend()

save_or_show(fig, "profile_vs_z")


# radial profiles
iphi = nearest_bin(phi_centers, RADIAL_PROFILE_PHI_RAD)
iz = nearest_bin(z_centers, RADIAL_PROFILE_Z_CM)

fig, ax = plt.subplots(figsize=(9, 6))
for name, array in components.items():
    if name == "Eabs":
        continue
    ax.plot(
        r_centers,
        array[:, iphi, iz],
        label=name,
    )

ax.set_xlabel("r [cm]")
ax.set_ylabel("Field [V/m]")
ax.set_title(
    rf"Field vs r at $\phi$={phi_centers[iphi]:.3f}, "
    rf"z={z_centers[iz]:.2f} cm"
)
ax.axhline(0.0, linewidth=1.0)
ax.grid(alpha=0.25)
ax.legend()

save_or_show(fig, "profile_vs_r")


# phi profiles
ir = nearest_bin(r_centers, PHI_PROFILE_R_CM)
iz = nearest_bin(z_centers, PHI_PROFILE_Z_CM)

fig, ax = plt.subplots(figsize=(9, 6))
for name, array in components.items():
    if name == "Eabs":
        continue
    ax.plot(
        phi_centers,
        array[ir, :, iz],
        label=name,
    )

ax.set_xlabel(r"$\phi$ [rad]")
ax.set_ylabel("Field [V/m]")
ax.set_title(
    rf"Field vs $\phi$ at r={r_centers[ir]:.2f} cm, "
    rf"z={z_centers[iz]:.2f} cm"
)
ax.axhline(0.0, linewidth=1.0)
ax.grid(alpha=0.25)
ax.legend()

save_or_show(fig, "profile_vs_phi")


## North–south side comparison


In [ ]:

positive_indices = np.where(z_centers > 0.0)[0]
negative_indices = np.where(z_centers < 0.0)[0]

n_pairs = min(
    len(positive_indices),
    len(negative_indices),
)

positive_indices = positive_indices[:n_pairs]
negative_indices = negative_indices[::-1][:n_pairs]

print("Paired z bins:", n_pairs)

for name, array in components.items():
    north = array[:, :, positive_indices]
    south_reflected = array[:, :, negative_indices]

    difference = north - south_reflected
    sum_field = north + south_reflected

    rms_difference = np.sqrt(
        np.mean(difference**2)
    )

    rms_sum = np.sqrt(
        np.mean(sum_field**2)
    )

    rms_field = np.sqrt(
        np.mean(array**2)
    )

    print(
        f"{name:5s}: "
        f"RMS(N-S)/RMS(field) = "
        f"{rms_difference / max(rms_field, 1e-30):.6f}; "
        f"RMS(N+S)/RMS(field) = "
        f"{rms_sum / max(rms_field, 1e-30):.6f}"
    )


## Boundary checks


In [ ]:

boundary_summary = {
    "z low": 0,
    "z high": -1,
}

for boundary_name, iz in boundary_summary.items():
    print(f"\nBoundary: {boundary_name}, z={z_centers[iz]:.3f} cm")

    for name, array in components.items():
        plane = array[:, :, iz]

        print(
            f"  {name:5s}: "
            f"mean={np.mean(plane): .6e}, "
            f"rms={np.sqrt(np.mean(plane**2)): .6e}, "
            f"maxabs={np.max(np.abs(plane)): .6e}"
        )


## Sector periodicity and Fourier content


In [ ]:

ir = nearest_bin(r_centers, PHI_PROFILE_R_CM)
iz = nearest_bin(z_centers, PHI_PROFILE_Z_CM)

for name in ("Er", "Ephi", "Ez", "Eabs"):
    values = components[name][ir, :, iz]

    fft = np.fft.rfft(values - np.mean(values))
    amplitudes = np.abs(fft)

    harmonics = np.arange(len(amplitudes))

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.stem(
        harmonics,
        amplitudes,
        basefmt=" ",
    )

    ax.set_xlim(0, min(24, len(amplitudes) - 1))
    ax.set_xlabel("Azimuthal harmonic m")
    ax.set_ylabel("Amplitude")
    ax.set_title(
        rf"Fourier content of {name} at "
        rf"r={r_centers[ir]:.2f} cm, z={z_centers[iz]:.2f} cm"
    )
    ax.grid(alpha=0.25)

    save_or_show(
        fig,
        f"fourier_{name}",
    )


## Charge-density QA


In [ ]:

if rho is None:
    print("Skipping charge QA because Charge3D/hRho is absent.")

else:
    rho_r_edges, rho_phi_edges, rho_z_edges = rho_edges

    rho_r_centers = 0.5 * (
        rho_r_edges[:-1] + rho_r_edges[1:]
    )

    rho_phi_centers = 0.5 * (
        rho_phi_edges[:-1] + rho_phi_edges[1:]
    )

    rho_z_centers = 0.5 * (
        rho_z_edges[:-1] + rho_z_edges[1:]
    )

    # r-phi slice
    iz = nearest_bin(rho_z_centers, Z_SLICE_CM)
    data = rho[:, :, iz]

    fig, ax = plt.subplots(figsize=(9, 6))
    mesh = ax.pcolormesh(
        rho_r_edges,
        rho_phi_edges,
        data.T,
        shading="auto",
    )
    fig.colorbar(mesh, ax=ax, label=r"$\rho$ [C/m$^3$]")
    ax.set_xlabel("r [cm]")
    ax.set_ylabel(r"$\phi$ [rad]")
    ax.set_title(
        rf"Charge density in $r$-$\phi$ at "
        rf"z={rho_z_centers[iz]:.2f} cm"
    )

    save_or_show(fig, "charge_rphi")


    # r-z slice
    iphi = nearest_bin(rho_phi_centers, PHI_SLICE_RAD)
    data = rho[:, iphi, :]

    fig, ax = plt.subplots(figsize=(9, 6))
    mesh = ax.pcolormesh(
        rho_r_edges,
        rho_z_edges,
        data.T,
        shading="auto",
    )
    fig.colorbar(mesh, ax=ax, label=r"$\rho$ [C/m$^3$]")
    ax.set_xlabel("r [cm]")
    ax.set_ylabel("z [cm]")
    ax.set_title(
        rf"Charge density in $r$-$z$ at "
        rf"$\phi$={rho_phi_centers[iphi]:.3f}"
    )

    save_or_show(fig, "charge_rz")


    # phi-z slice
    ir = nearest_bin(rho_r_centers, R_SLICE_CM)
    data = rho[ir, :, :]

    fig, ax = plt.subplots(figsize=(9, 6))
    mesh = ax.pcolormesh(
        rho_phi_edges,
        rho_z_edges,
        data.T,
        shading="auto",
    )
    fig.colorbar(mesh, ax=ax, label=r"$\rho$ [C/m$^3$]")
    ax.set_xlabel(r"$\phi$ [rad]")
    ax.set_ylabel("z [cm]")
    ax.set_title(
        rf"Charge density in $\phi$-$z$ at "
        rf"r={rho_r_centers[ir]:.2f} cm"
    )

    save_or_show(fig, "charge_phiz")


    # Integrated radial, phi, and z profiles.
    dr = np.diff(rho_r_edges)
    dphi = np.diff(rho_phi_edges)
    dz = np.diff(rho_z_edges)

    radial_weight = rho_r_centers[:, None, None]

    cell_volume = (
        radial_weight
        * dr[:, None, None]
        * dphi[None, :, None]
        * dz[None, None, :]
    )

    charge_per_cell = rho * cell_volume

    charge_vs_r = np.sum(
        charge_per_cell,
        axis=(1, 2),
    )

    charge_vs_phi = np.sum(
        charge_per_cell,
        axis=(0, 2),
    )

    charge_vs_z = np.sum(
        charge_per_cell,
        axis=(0, 1),
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(rho_r_centers, charge_vs_r)
    ax.set_xlabel("r [cm]")
    ax.set_ylabel("Integrated charge [C]")
    ax.set_title("Integrated charge vs radius")
    ax.grid(alpha=0.25)
    save_or_show(fig, "charge_vs_r")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(rho_phi_centers, charge_vs_phi)
    ax.set_xlabel(r"$\phi$ [rad]")
    ax.set_ylabel("Integrated charge [C]")
    ax.set_title("Integrated charge vs phi")
    ax.grid(alpha=0.25)
    save_or_show(fig, "charge_vs_phi")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(rho_z_centers, charge_vs_z)
    ax.set_xlabel("z [cm]")
    ax.set_ylabel("Integrated charge [C]")
    ax.set_title("Integrated charge vs z")
    ax.grid(alpha=0.25)
    save_or_show(fig, "charge_vs_z")

    print("Integrated charge [C]:", np.sum(charge_per_cell))


## Potential QA


In [ ]:

if Phi is None:
    print("Skipping potential QA because Field3D/hPhi is absent.")

else:
    iz = nearest_bin(z_centers, Z_SLICE_CM)

    fig, ax = plt.subplots(figsize=(9, 6))
    data = Phi[:, :, iz]

    mesh = ax.pcolormesh(
        r_edges,
        phi_edges,
        data.T,
        shading="auto",
    )

    fig.colorbar(mesh, ax=ax, label=r"$\Phi$ [V]")
    ax.set_xlabel("r [cm]")
    ax.set_ylabel(r"$\phi$ [rad]")
    ax.set_title(
        rf"Potential in $r$-$\phi$ at z={z_centers[iz]:.2f} cm"
    )

    save_or_show(fig, "potential_rphi")


## Compact numerical summary


In [ ]:

print("\nField summary")
print("=" * 72)

for name, array in components.items():
    finite = array[np.isfinite(array)]

    print(
        f"{name:5s} "
        f"min={np.min(finite): .6e} "
        f"max={np.max(finite): .6e} "
        f"mean={np.mean(finite): .6e} "
        f"rms={np.sqrt(np.mean(finite**2)): .6e}"
    )

print("\nNon-finite counts")
print("=" * 72)

for name, array in components.items():
    print(
        f"{name:5s}: "
        f"{np.size(array) - np.count_nonzero(np.isfinite(array))}"
    )

print("\nPlots written to:", OUTPUT_DIR.resolve())
